# Backtest v2 — Reproducibility Notebook

Drives the full Phase 5 pipeline and renders the paper figures.

**Spec:** [`specs/03_backtest_v2.md`](../specs/03_backtest_v2.md)

**Inputs (built by earlier phases):**
- `data/pit_universe.csv` — PIT universe from Phase 1
- `data/binance_usdt_pairs_2018-12-31_2024-01-01_1d.csv` + `data/binance_pit_supplement_2019-2024_1d.csv` — price panels

**Outputs:** all `data/backtest_v2_*.csv` + `data/inference_*.csv` + `paper/figures/*.png`.

**Runtime:** ~3 minutes on the full 11-strategy headline run.

## 1. Imports & data load

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

from src.universe import load_pit_universe
from src.backtest import WalkForwardBacktest, COST_SCENARIOS, summarize_performance
from src.portfolio_maker import (
    HRP, HRPDetoned, HRPPartialCorr, HRPTailDep, HRPTailDepShrunk,
    HRPShrunkCov, HRPVolStd, IVP, MVP, ERC, MaxDiv, NetworkRiskParity,
)
from src.inference import (
    stationary_block_bootstrap, sharpe_diff_ledoit_wolf, hansen_spa_test,
)

sns.set_style('whitegrid')

pit = load_pit_universe('../data/pit_universe.csv')
existing = pd.read_csv('../data/binance_usdt_pairs_2018-12-31_2024-01-01_1d.csv', parse_dates=['open_time'])
supp = pd.read_csv('../data/binance_pit_supplement_2019-2024_1d.csv', parse_dates=['open_time'])
prices_long = pd.concat([existing, supp], ignore_index=True).drop_duplicates(subset=['symbol','open_time'])
prices_long['close'] = prices_long['close'].astype(float)
prices = prices_long.pivot(index='open_time', columns='symbol', values='close').sort_index()
print(f'price panel: {prices.shape}, PIT snapshots: {pit["date"].nunique()}')

## 2. Scenario B (monthly rebal, Conservative CEX cost)

Headline strategies. For the full 19-strategy comparison see `data/backtest_v2_rebalance_results.csv`.

In [ ]:
START, END = '2020-01-01', '2026-05-18'
cost = COST_SCENARIOS['conservative_cex']
factories = {
    'HRP':               lambda r: HRP(r),
    'HRP_Detoned':       lambda r: HRPDetoned(r),
    'HRP_TailDep':       lambda r: HRPTailDep(r, q=0.05),
    'HRP_TailDepShrunk': lambda r: HRPTailDepShrunk(r, q=0.05),
    'HRP_ShrunkCov':     lambda r: HRPShrunkCov(r),
    'MVP':               lambda r: MVP(r),
    'ERC':               lambda r: ERC(r),
}
returns_dict = {}
for name, factory in factories.items():
    res = WalkForwardBacktest(prices, pit, factory, cost_model=cost).run(START, END)
    returns_dict[name] = res['daily_returns']
returns_dict['HODL_BTC'] = prices['BTCUSDT'].loc[START:END].pct_change().dropna()
rdf = pd.DataFrame(returns_dict).dropna()
for name in rdf.columns:
    m = summarize_performance(rdf[name], name=name)
    print(f"  {name:18s}  Sharpe={m['sharpe']:+.3f}  TotRet={m['total_return']:+.1%}  MaxDD={m['max_drawdown']:.1%}")

## 3. Cumulative returns (equity curves)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
cum = (1 + rdf).cumprod()
for c in cum.columns:
    style = '--' if c in ('HODL_BTC', 'MVP') else '-'
    ax.plot(cum.index, cum[c], style, label=c, linewidth=1.4)
ax.set_yscale('log')
ax.set_ylabel('Cumulative return (log scale)')
ax.set_title('Scenario B equity curves, 2020-01 → 2026-05')
ax.legend(loc='upper left', ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Statistical inference

- Bootstrap CIs for Sharpe (Spec 09 §1)
- LW pairwise diff vs HRP (Spec 09 §2)
- Hansen SPA across all candidates (Spec 09 §3)

In [ ]:
def sharpe(r):
    r = pd.Series(r) if not isinstance(r, pd.Series) else r
    s = float(r.std(ddof=1))
    return float(r.mean() / s * np.sqrt(365)) if s > 0 else 0.0

# Bootstrap CIs for HRP and the headline variants
for s in ['HRP', 'HRP_TailDepShrunk', 'HRP_ShrunkCov', 'MVP', 'HODL_BTC']:
    boot = stationary_block_bootstrap(rdf[s].values, sharpe, n_iter=1000, seed=42)
    print(f"  {s:20s}  Sharpe={boot['point_estimate']:+.3f}  CI[{boot['ci_lower_95']:+.3f}, {boot['ci_upper_95']:+.3f}]")

In [ ]:
# Hansen SPA
spa = hansen_spa_test(rdf.drop(columns=['HRP']), rdf['HRP'].values, n_bootstrap=500, seed=42)
spa.round(3)

## 5. Pre-generated figures

All paper figures live in `paper/figures/`. Built by `scripts/generate_paper_figures.py`:
- `rebalance_cumulative_returns.png`
- `risk_return_scatter.png`
- `metrics_heatmap.png`
- `cluster_stability_timeseries.png`
- `spa_pvalues_barplot.png`
- `cost_sensitivity_sharpe.png`
- `detoning_vs_partial_corr.png`
- `shrinkage_intensity_timeseries.png`
- `weight_concentration.png`
- `turnover_analysis.png`
- `pit_universe_evolution.png`

In [ ]:
from IPython.display import Image, display
for fname in ['rebalance_cumulative_returns', 'risk_return_scatter', 'metrics_heatmap',
              'cluster_stability_timeseries', 'spa_pvalues_barplot', 'cost_sensitivity_sharpe',
              'detoning_vs_partial_corr', 'weight_concentration', 'turnover_analysis']:
    print(f'\n{fname}.png')
    display(Image(f'../paper/figures/{fname}.png'))